In [0]:
df = spark.table("silver_aapl_features")
pd_df = df.toPandas()

pd_df = pd_df.sort_values("Date")
pd_df.head()

In [0]:
feature_cols = [
    "ma_5",
    "ma_10",
    "ma_20",
    "lag_1",
    "lag_3",
    "lag_7"
]

X = pd_df[feature_cols]
y = pd_df["target"]

In [0]:
import mlflow

run_id = "521aac70f3234b00b77f9c3249dec68b"

model_uri = f"runs:/{run_id}/model"

model = mlflow.pyfunc.load_model(model_uri)

In [0]:
pd_df["pred_target"] = model.predict(X)

In [0]:
gold_pd = pd_df[
    [
        "Date",
        "ticker",
        "Close",
        "pred_target"
    ]
].copy()

gold_pd = gold_pd.rename(
    columns={
        "Date": "prediction_date",
        "Close": "close_price",
        "pred_target": "pred_direction"
    }
)

In [0]:
gold_df = spark.createDataFrame(gold_pd)

gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_stock_predictions")

In [0]:
display(spark.table("gold_stock_predictions"))